In [1]:
import pandas as pd
import os
import re

In [2]:
postings_df = pd.read_csv('data/postings.csv')
postings_df.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [3]:
postings_df.shape

(123849, 31)

In [4]:
def preprocess_text(text):
    """Preprocess text by converting to lowercase, removing special characters, and handling NaN."""
    if pd.isnull(text):
        return ""
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove special characters
    return text.strip()

def combine_features(row):
    """Combine relevant features into a single string."""
    features = []
    for col in ['title', 'description', 'skills_desc']:
        if not pd.isnull(row[col]):
            features.append(f"{col.capitalize()}: {preprocess_text(row[col])}\n")
    return ' '.join(features)

# Apply preprocessing and feature combination
postings_df['combined_features'] = postings_df.apply(combine_features, axis=1)

# Display a sample of the combined data
postings_df[['job_id','combined_features']].head()


,job_id,combined_features
0,921716,Title: marketing coordinator\n Description: jo...
1,1829192,Title: mental health therapistcounselor\n Desc...
2,10998357,Title: assitant restaurant manager\n Descripti...
3,23221523,Title: senior elder law trusts and estates as...
4,35982263,Title: service technician\n Description: looki...


In [5]:
resume_df = pd.read_csv('data/Resume/Resume.csv')
resume_df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [6]:
def preprocess_resume_text(row):
    """Preprocess resume text by converting to lowercase, removing special characters, and handling NaN."""
    text = row.get('Resume_str', '') 
    if pd.isnull(text):
        return ""
    text = re.sub(r'[^\w\s,+./-]', '', text)  # Remove unwanted characters but retain important ones like +, , . / -
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespaces
    text = text.strip()  # Trim leading/trailing whitespace
    text = text.lower() # Normalize to lowercase
    
    return text

# Apply preprocessing
resume_df['preprocessed_resume'] = resume_df.apply(preprocess_resume_text, axis=1)

# Display the first 5 rows
resume_df[['ID', 'preprocessed_resume']].head()

,ID,preprocessed_resume
0,16852973,hr administrator/marketing associate hr admini...
1,22323967,"hr specialist, us hr operations summary versat..."
2,33176873,hr director summary over 20 years experience i...
3,27018550,"hr specialist summary dedicated, driven, and d..."
4,17812897,hr manager skill highlights hr skills hr depar...


In [7]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load pre-trained BERT model and tokenizer
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Prepare inputs
resume = 
job_descriptions = postings_df['combined_features'].tolist()
similarity_scores = []
i = 0
# Loop through job descriptions
for job_description in job_descriptions[:100]:
    # Prepare inputs for each pair
    i+=1
    inputs = tokenizer(resume, job_description, return_tensors="pt", truncation=True, padding=True)
    print(i)
    # Forward pass
    outputs = model(**inputs)
    similarity_score = torch.sigmoid(outputs.logits)  # Convert logits to probabilities
    similarity_scores.append(similarity_score.item())

# Print all similarity scores
print(similarity_scores)

1


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


2
3
4


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


5
6


...

95


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


96
97
98
99


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


100
[0.3577868640422821, 0.5779439806938171, 0.5202832818031311, 0.4788808822631836, 0.38102900981903076, 0.4582471549510956, 0.4102717339992523, 0.3958040773868561, 0.5569345355033875, 0.5839624404907227, 0.39092275500297546, 0.3471144437789917, 0.5269433259963989, 0.36054542660713196, 0.5720700621604919, 0.5460296273231506, 0.5047323107719421, 0.541833221912384, 0.4059792459011078, 0.3812800645828247, 0.6020666360855103, 0.43225550651550293, 0.3266446888446808, 0.37533146142959595, 0.49604469537734985, 0.5689465999603271, 0.320010244846344, 0.3748953342437744, 0.35361865162849426, 0.5598855018615723, 0.4235334098339081, 0.4668954014778137, 0.376699298620224, 0.3956674039363861, 0.596331775188446, 0.38355857133865356, 0.33278486132621765, 0.6060372591018677, 0.4438832998275757, 0.37617623805999756, 0.43970778584480286, 0.4766274690628052, 0.4571988582611084, 0.4205003082752228, 0.47851797938346863, 0.39753007888793945, 0.6078213453292847, 0.3903852701187134, 0.4312334656715393, 0.4868

In [10]:
import numpy as np
i = np.argmax(similarity_scores)
print(job_descriptions[i])



Title: manager retail pharmacy
 Description: summarymanages operation and supervises all departmental distributionclinical and educational activities plans controls coordinates and measures the work of the departmentessential functionsmanages staff interviews hires and trains evaluates employee performance deals with performance problems as appropriate delegates work assignments effectivelyassists in managing department budgetmanages pharmacy operations and coordinates functions with the needs of other departments oversees and manages drug purchases information and review for drug interactionsbenchmarks pharmacy operations with localregional and national solutionscritically reviews the medical literature collates and summarizes studies and makes recommendations to the appropriate partynetworks with hospital departments takes input and in conjunction with administration and pharmacy department to develop projects and monitors their progress to completionmonitors pharmacy payment methodo

In [14]:
from torch.utils.data import DataLoader

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define batch size and initialize variables
batch_size = 32
max_score = float('-inf')
best_job_index = -1

# Create DataLoader for batching
data_loader = DataLoader(job_descriptions[:10000], batch_size=batch_size, shuffle=False)

j=0
# Iterate through batches
for batch_start, batch in enumerate(data_loader):
    # Tokenize and move inputs to GPU
    inputs = tokenizer([resume] * len(batch), list(batch), return_tensors="pt", 
                       truncation=True, padding=True).to(device)
    j+=32
    print(j)
    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Compute similarity scores
    batch_scores = torch.sigmoid(outputs.logits).squeeze().tolist()
    
    # Update maximum score and index
    for i, score in enumerate(batch_scores):
        global_index = batch_start * batch_size + i  # Compute global index
        if score > max_score:
            max_score = score
            best_job_index = global_index

# Retrieve the job with the highest similarity score
best_job_description = job_descriptions[best_job_index]
print(f"Highest Similarity Score: {max_score}")
print(f"Best Job Description: {best_job_description}")

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

32


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

64


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

96


KeyboardInterrupt: 